# Case Study: Cyclistic Bike-Share

**How do annual members and casual riders use Cyclistic bikes differently?**<br>
*Google Data Analytics Professional Certificate – Course 8 Capstone*

**Shaine Meister**<br>
**March 28, 2026**

---


## Introduction
Cyclistic is a bike-share program in Chicago with more than 5,800 bicycles and 600 docking stations. The company offers traditional bikes as well as assistive options such as reclining bikes, hand tricycles, and cargo bikes. Until recently, Cyclistic’s marketing strategy focused on building general awareness with flexible pricing: single-ride passes, full-day passes, and annual memberships. Casual riders buy single-ride or full-day passes; annual members purchase yearly memberships.

Cyclistic’s finance team has shown that annual members are far more profitable than casual riders. The director of marketing, Lily Moreno, believes the company’s future success depends on converting casual riders into annual members. To design an effective marketing campaign, the team first needs clear answers to three guiding questions:

1. **How do annual members and casual riders use Cyclistic bikes differently?**
2. **Why would casual riders buy Cyclistic annual memberships?**  
3. **How can Cyclistic use digital media to influence casual riders to become members?**

This notebook follows the official Google 6-step data analysis process (**Ask**, **Prepare**, **Process**, **Analyze**, **Share**, & **Act**) exactly. All analysis is performed in Python using only **pandas** and **numpy** modules. The data is the most recent 12 months of Cyclistic trip records (public Divvy dataset, March 2025 – February 2026). No personally identifiable information is present.

## 1. Ask
**Business task**  
Design marketing strategies aimed at converting casual riders into annual members. The first step is to understand how annual members and casual riders use Cyclistic bikes differently so that targeted campaigns can be built.

**Key stakeholders**  
- Lily Moreno – Director of Marketing  
- Cyclistic marketing analytics team  
- Cyclistic executive team  

**Guiding questions (primary focus of this analysis)**  
- How do annual members versus casual riders differ in usage?  
- Why would casual riders buy a membership?  
- How can digital media convert casual riders into members?  

This analysis directly answers the first question and provides the foundation for answering the other two.

**Success criteria**  
Identify clear, actionable differences in ride length, day-of-week patterns, rideable type usage, and station preferences between the two rider groups.

## 2. Prepare
**Data location**  
Public Cyclistic (Divvy) trip data: [divvy data link](https://divvy-tripdata.s3.amazonaws.com/index.html "https://divvy-tripdata.s3.amazonaws.com/index.html")<br>
Weather Data: [Open-Meteo's](https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv "https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv")<br>
In this section, the notebook builds the raw trip dataset by validating the requested monthly Divvy files against the public S3 bucket, downloading ZIP archives when needed, extracting the monthly CSV files, and combining them into a single DataFrame.

**What this section does**
1. Validate the configured `start_yyyymm` to `end_yyyymm` month range against available Divvy S3 objects.
2. Download missing ZIP archives to a local cache and extract their CSV contents.
3. Concatenate all monthly CSV files into one consolidated dataset.
4. Standardize `started_at` as datetime and sort records chronologically.
5. Run a quick schema sanity check across extracted monthly files.

**How the data is organized** *(sample)* 
<div style="font-size: 10px;">

| `ride_id` | `rideable_type` | `started_at` | `ended_at` | `start_station_name` | `start_station_id` | `end_station_name` | `end_station_id` |
|---|---|---|---|---|---|---|---|
| Unique trip identifier | Bike type used | Trip start timestamp | Trip end timestamp | Origin station name | Origin station ID | Destination station name | Destination station ID |

</div>

The combined trip dataset is expected to contain approximately 5-6 million rows, depending on the selected date range.

**ROCCC verification**  
- Reliable: Collected by Cyclistic's own system.  
- Original: First-party trip data.  
- Comprehensive: Covers every ride in the system.  
- Current: Uses the configured recent month range.  
- Cited: Licensed for analysis (Motivate International).  

**Licensing, privacy, and security**  
Data is public under the Divvy data license. No personally identifiable information is included, so privacy is preserved. Files are cached locally for reproducible reruns of the notebook.

**Prepare outputs produced here**  
- `df`: consolidated raw trip dataset with `started_at` parsed and sorted.  
- `schema_check`: quick cross-file schema QA summary.  

**Data integrity check**  
A quick preview of row structure and column consistency is displayed below before the notebook moves to the next section.

In [ ]:
# Standard library imports
# calendar: monthrange helper; derives the last day of any month for dynamic weather date bounds
# io: in-memory byte/string stream; wraps weather CSV text for pd.read_csv
# os: file-system path utilities (available for downstream cells if needed)
# re: regular expressions; extracts ZIP filenames from the S3 XML bucket listing
# zipfile: ZIP archive reading and member extraction for Divvy monthly trip files
# pathlib.Path: object-oriented path handling for local cache directories
# urllib.request: urlopen streams HTTP for S3 XML and weather CSV; urlretrieve downloads ZIPs to disk

# Third-party libraries
# numpy: array operations for boolean dirty-data masks and coordinate grid math
# pandas: core DataFrame library for trip records, weather tables, and grouped output

import calendar
import io
import os
import re
import zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import numpy as np
import pandas as pd

# Shared path configuration (define once and reuse throughout notebook).
PROJECT_ROOT = Path("/home/stubb/Code/articles/case-study_bike-share-success")
OUTPUT_DIR = PROJECT_ROOT

# Configure optional Excel export (CSV is always exported).
export_xlsx = False  # Set to True to also write an .xlsx file.

## 3. Process
**Tools of choice**<br>
Python with pandas and numpy - scalable, reproducible, and suitable for large trip data and downstream enrichment workflows.

This section covers the full processing pipeline applied after the raw trip files are prepared. It starts with Divvy trip validation and feature engineering, removes rule-flagged bad records, converts raw coordinates into stable vector-mapped location keys, enriches trips with nearest-station hourly weather data, and finishes by exporting the final enriched dataset for downstream analysis.

**Workflow Summary:**
1. **Divvy Data Processing**
   - Standardize datetime fields, calculate `ride_length`, round timestamps, and derive reusable time features.
   - Build reusable validation masks and dirty-data rules for chronology, missing values, and duplicate records.
   - Generate QA tables, A/B comparisons, and reconciliation summaries.

2. **Bad Data Clean Up and Drop**
   - Combine all rule flags into one dirty-row mask.
   - Split records into `dirty_data` and `clean_data`, remove invalid rows from the working dataset, and drop no-longer-needed columns.

3. **Coordinate Vector Mapping**
   - Build a geographic bounding box from trip coordinates.
   - Map start and end latitude/longitude values into stable grid-based vector keys for downstream grouping and comparison.

4. **Weather Data Processing and Merge**
   - Match each trip start grid point to the nearest weather station.
   - Merge hourly weather observations into trip records and produce merge-audit diagnostics.

5. **Export Final Enriched Data**
   - Export the final processed dataset as CSV, with optional Excel output.

**Primary outputs from this section**
- `clean_data` and `dirty_data` for traceable record separation.  
- Vector-mapped trip fields for location-based grouping.  
- `df_weather_merge` as the final weather-enriched analytical dataset.  
- QA and audit tables that document validation and merge quality.

## 4. Analyze

**Key calculations performed**
- Average ride length by rider type
- Number of rides by day of week and rider type
- Rideable type usage
- Top stations for each group#

In [9]:
# Core summary statistics
summary = df.groupby('member_casual').agg(
    avg_ride_length=('ride_length', 'mean'),
    ride_count=('ride_id', 'count'),
    max_ride_length=('ride_length', 'max')
).round(2)

print(summary)

# Rides by day of week
day_summary = df.groupby(['day_of_week', 'member_casual']).size().unstack()
day_summary.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print(day_summary)

# Rideable type usage
rideable_summary = df.groupby(['member_casual', 'rideable_type']).size().unstack()
print(rideable_summary)

               avg_ride_length  ride_count  max_ride_length
member_casual                                              
casual                   22.63     2013503          1574.90
member                   12.43     3588130          1499.97
member_casual  casual  member
Mon            230447  506390
Tue            228150  574502
Wed            223219  559468
Thu            258886  578581
Fri            322828  534018
Sat            415983  452449
Sun            333990  382722
rideable_type  classic_bike  electric_bike
member_casual                             
casual               676155        1337348
member              1280363        2307767
